In [2]:
import cv2
import mediapipe as mp
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional,LSTM, Dense,Input,Flatten, GRU

from glob import glob
import os
from pathlib import Path

from tqdm.notebook import tqdm

from numpy import  argmax, array, expand_dims 
import numpy as np

import utils


In [ ]:
# Initialize Mediapipe Pose model
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils  
pose = mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7)


mp_utils = utils.MediapipeUtils(mp_pose, mp_drawing)

In [3]:
# Specify the video file(s)
DATA_DIR = glob("Dataset/*/*/*.mp4")
DATA_PATH = "MP_Data"

# Landmarks to extract (based on Mediapipe Pose Landmarks)
# landmark_indices = [11, 12, 13, 14, 15, 16]

angle_landmarks = ((12,14),
                   (14,16),
                   (11,13),
                   (13,15))

labels = [i.split("\\")[-1] for i in glob("Dataset\*\*")]

labels

['Blind',
 'Deaf',
 'Flat',
 'Happy',
 'Poor',
 'Quiet',
 'Rich',
 'sad',
 'Slow',
 'Thick']

In [5]:
try:
    # os.mkdir('MP_data')
    for i in range(len(labels)):
        os.mkdir(f'MP_data/{labels[i]}')
except:
    print("Directory already exists")

Directory already exists


In [ ]:
# Process each video
for video_path in tqdm(DATA_DIR):
    data = []
    cap = cv2.VideoCapture(video_path)
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert the frame to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Process the image and detect pose landmarks
        results = pose.process(image)

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # Extract X, Y, Z coordinates for specified landmarks
            # keypoints = []
            angles_keypoint = []
            # for idx in landmark_indices:
            #     lm = landmarks[idx]
            #     keypoints.extend([lm.x, lm.y, lm.z])

            for idx in angle_landmarks:
                p1 = landmarks[idx[0]]                
                p2 = landmarks[idx[1]]    
                
                vector = np.array([p2.x - p1.x, p2.y - p1.y, p2.z - p1.z])
                            
                ang = mp_utils.calculate_angles(vector)
                angles_keypoint.extend(ang)
            
            # Calculate angles between vectors
            # Left arm: vectors 15-13 and 13-11
            left_wrist = np.array([landmarks[15].x, landmarks[15].y, landmarks[15].z])
            left_elbow = np.array([landmarks[13].x, landmarks[13].y, landmarks[13].z])
            left_shoulder = np.array([landmarks[11].x, landmarks[11].y, landmarks[11].z])

            vector_left_wrist_elbow = left_wrist - left_elbow
            vector_left_elbow_shoulder = left_elbow - left_shoulder

            angle_left_arm = mp_utils.calculate_angles_between(vector_left_wrist_elbow, vector_left_elbow_shoulder)

            # Right arm: vectors 16-14 and 14-12
            right_wrist = np.array([landmarks[16].x, landmarks[16].y, landmarks[16].z])
            right_elbow = np.array([landmarks[14].x, landmarks[14].y, landmarks[14].z])
            right_shoulder = np.array([landmarks[12].x, landmarks[12].y, landmarks[12].z])

            vector_right_wrist_elbow = right_wrist - right_elbow
            vector_right_elbow_shoulder = right_elbow - right_shoulder

            angle_right_arm = mp_utils.calculate_angles_between(vector_right_wrist_elbow, vector_right_elbow_shoulder)
            
            # Distance between points 15 and 16 (hands)
            left_hand = landmarks[15]
            right_hand = landmarks[16]
            distance_hands = mp_utils.calculate_distance(left_hand, right_hand)
            
            # Combine all features
            features = angles_keypoint + [angle_left_arm, angle_right_arm, distance_hands]
            data.append(features)

    data = np.array(data)
    path = video_path.split("/")
    
    # Getting Current action of the video
    action = video_path.split("\\")[-2]
    
    #Get the sequence number
    sequence = video_path.split("\\")[-1][:-4]
    
    npy_path = os.path.join(DATA_PATH, action, sequence)
    print(npy_path)
    
    np.save(npy_path, data)

    cap.release()

pose.close()


  0%|          | 0/110 [00:00<?, ?it/s]

MP_Data\Blind\1-1
MP_Data\Blind\1-2
MP_Data\Blind\1-3
MP_Data\Blind\1-4
MP_Data\Blind\1-5
MP_Data\Blind\1-6
MP_Data\Blind\1-7
MP_Data\Blind\1-8
MP_Data\Deaf\Deaf-1
MP_Data\Deaf\Deaf-2
MP_Data\Deaf\Deaf-3
MP_Data\Deaf\Deaf-4
MP_Data\Deaf\Deaf-5
MP_Data\Deaf\Deaf-6
MP_Data\Deaf\Deaf-7
MP_Data\Deaf\Deaf-8
MP_Data\Flat\Flat-1
MP_Data\Flat\Flat-2
MP_Data\Flat\Flat-3
MP_Data\Flat\Flat-4
MP_Data\Flat\Flat-5
MP_Data\Flat\Flat-6
MP_Data\Flat\Flat-7
MP_Data\Flat\Flat-8
MP_Data\Happy\Happy-1
MP_Data\Happy\Happy-10
MP_Data\Happy\Happy-11
MP_Data\Happy\Happy-12
MP_Data\Happy\Happy-13
MP_Data\Happy\Happy-14
MP_Data\Happy\Happy-15
MP_Data\Happy\Happy-2
MP_Data\Happy\Happy-3
MP_Data\Happy\Happy-4
MP_Data\Happy\Happy-5
MP_Data\Happy\Happy-6
MP_Data\Happy\Happy-7
MP_Data\Happy\Happy-8
MP_Data\Happy\Happy-9
MP_Data\Poor\Poor-1
MP_Data\Poor\Poor-2
MP_Data\Poor\Poor-3
MP_Data\Poor\Poor-4
MP_Data\Poor\Poor-5
MP_Data\Poor\Poor-6
MP_Data\Poor\Poor-7
MP_Data\Poor\Poor-8
MP_Data\Quiet\quiet-1
MP_Data\Quiet\quie

In [ ]:




# Labels for data
actions = array([i.split("\\")[-1] for i in glob('MP_Data\*')])
# actions = array(['Beautiful','Blind','Deaf','happy','loud','quiet','sad','Ugly'])
# actions = array(['Bank', 'big large', 'Bird', 'Black', 'Boy', 'Brother', 'Car', 'Cell phone', 'Court', 'Cow', 'Death', 'Dog', 'dry', 'Election', 'Fall', 'Fan', 'Father', 'Girl', 'good', 'Good Morning', 'happy', 'Hat', 'Hello', 'hot', 'House', 'I', 'it', 'long', 'loud', 'Monday', 'new', 'Paint', 'Pen', 'Priest', 'quiet', 'Red', 'Shoes', 'short', 'small little', 'Store or Shop', 'Summer', 'T-Shirt', 'Teacher', 'Thank you', 'Time', 'train ticket', 'White', 'Window', 'Year', 'you (plural)'])

# Defining Model
  
# num_classes =  50

# input_shape = (max_frames, 132)
# num_classes =  8

max_frames = 28
input_shape = (max_frames, 15)
num_classes =  len(actions)
angle_landmarks = ((12,14),
                   (14,16),
                   (11,13),
                   (13,15))

# Loading Model    
model = Sequential([        
        Input(shape=input_shape),        
        
        GRU(64, return_sequences=True),
        GRU(128, return_sequences=True),
        GRU(64, return_sequences=True),
        
        # Flatten the output
        Flatten(),
        
        # Fully connected layer
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(num_classes, activation='softmax')
])

model_path = Path.cwd() / 'Model' / 'INCLUDE_8_V4_angles.h5'
model.load_weights(str(model_path))


# sequence = [[0] * 258] * (max_frames // 2) # Passing an enpty list of 258 elements to start the prediction from as soon as the input feed starts

n_frames = 0
sequence = [[0] * 15] * (max_frames // 2) 
sentence = []
threshold = 0.9

cap = cv2.VideoCapture(0) # Default camera
# cap = cv2.VideoCapture("Test Recordings\\test (5).mp4")
# cap = cv2.VideoCapture("Dataset\Adjectives\\7. Deaf\MVI_9583.mp4")


with mp_holistic.Holistic(min_detection_confidence=0.7, 
                          min_tracking_confidence=0.7) as holistic:
    while cap.isOpened():    
        ret, frame = cap.read()

        # Extract X, Y, Z coordinates for specified landmarks
        data = []
        angles_keypoint = []
        
        # Make detections
        image, results = mp_utils.mediapipe_detection(frame, holistic)

        mp_utils.draw_styled_landmarks(image, results)
        
        if results != None:
            results = results.pose_landmarks.landmark

            for idx in angle_landmarks:
                p1 = results[idx[0]]                
                p2 = results[idx[1]]    
                
                vector = np.array([p2.x - p1.x, p2.y - p1.y, p2.z - p1.z])
                            
                ang = calculate_angles(vector)
                angles_keypoint.extend(ang)
            
            # Calculate angles between vectors
            # Left arm: vectors 15-13 and 13-11
            left_wrist = np.array([results[15].x, results[15].y, results[15].z])
            left_elbow = np.array([results[13].x, results[13].y, results[13].z])
            left_shoulder = np.array([results[11].x, results[11].y, results[11].z])

            vector_left_wrist_elbow = left_wrist - left_elbow
            vector_left_elbow_shoulder = left_elbow - left_shoulder

            angle_left_arm = calculate_angles_between(vector_left_wrist_elbow, vector_left_elbow_shoulder)

            # Right arm: vectors 16-14 and 14-12
            right_wrist = np.array([results[16].x, results[16].y, results[16].z])
            right_elbow = np.array([results[14].x, results[14].y, results[14].z])
            right_shoulder = np.array([results[12].x, results[12].y, results[12].z])

            vector_right_wrist_elbow = right_wrist - right_elbow
            vector_right_elbow_shoulder = right_elbow - right_shoulder

            angle_right_arm = calculate_angles_between(vector_right_wrist_elbow, vector_right_elbow_shoulder)
            
            # Distance between points 15 and 16 (hands)
            left_hand = results[15]
            right_hand = results[16]
            distance_hands = calculate_distance(left_hand, right_hand)
            
            # Combine all features
            features = angles_keypoint + [angle_left_arm, angle_right_arm, distance_hands]
            # data.append(features)
            
            sequence.append(features)    
                    
        # Predicting output in every 10 frames
        if n_frames % 5 == 0:
            sequence = sequence[-max_frames:]
                    
            # 2. Prediction logic
            if len(sequence) == max_frames:
            
                res = model.predict(expand_dims(sequence, axis=0))[0]
                # res = model.predict(sequence)   
                print(actions[np.argmax(res)], res[argmax(res)])

                # 3. Text Script
                if res[argmax(res)] > threshold:
                    if len(sentence) > 0:
                        if actions[argmax(res)] != sentence[-1]:
                            sentence.append(actions[argmax(res)])
                    else:
                        sentence.append(actions[argmax(res)])
                
                # res = res.remove(res[argmax(res)])
                # print(actions[np.argmax(res)], res[argmax(res)])
                
            if len(sentence) > 5:
                sentence = sentence[-5:]

        cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, ' '.join(sentence), (3, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.imshow('OpenCV Feed', image)
        n_frames += 1

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
        
    print(sentence)
    cap.release()
    cv2.destroyAllWindows()

AttributeError: 'NoneType' object has no attribute 'landmark'